# Field validation — `native_fields` (DEPTH pipeline)

| | |
|---|---|
| Subset | `native_fields` |
| Pipeline | DEPTH |
| Timestep | 2012-11-09 12:00:00 |
| Domain | one 720 × 720 × 51 tile (≈1400 × 1400 km), set in Section 1 |
| Depth levels | `sfc`, `z25m`, `mld`, `mld_mean` |
| Data | computed on the fly from `s3://dbof/LLC4320_RAW/DEPTH/` |
| Plan | `prompts/field_validation_depth.md` |
| Field reference | `docs/Fields.md` |

Rows of the map and PDF figures are **depth levels**, not regions — that
is the one structural difference from the surface notebooks.

The simplest depth subset: raw model variables at each depth level, with one real computation — U and V are staggered and in MODEL x/y, so they are interpolated to tracer points and rotated to east/north.  The figures show the model-axis components beside the rotated ones so the rotation is visible rather than assumed.

## Section 1 — Setup

Everything configurable is in the next cell: the **region**, the date,
the depth levels, and the zoom size.  Change `REGION` to validate a
different part of the ocean — any key in `dbof.plotting.regions.REGIONS`
that carries a `zoom` anchor.

Default is the Gulf Stream, anchored at 60°W / 37°N — dynamically
active in every field this project computes, and the same point the
surface notebooks zoom into, so surface and depth look at the same
water.


In [ ]:
# ---- knobs -------------------------------------------------------------
REGION       = "gulf_stream"          # any region with a 'zoom' anchor
DATE         = "2012-11-09 12:00:00"  # the only DEPTH date transferred so far
LEVELS       = ("sfc", "z25m", "mld", "mld_mean")
ZOOM_HALF_KM = 100.0                  # -> a 200 x 200 km zoom box

SUBSET   = "native_fields"
PIPELINE = "DEPTH"
RAW_VARS = ["Theta", "Salt", "Eta", "U", "V", "W"]

# Profiles (Figure 3)
N_PROFILES        = 5       # <= 5; the fixed location colours are not cycled
PROFILE_SEED      = 42      # same 5 columns for every field in the notebook
PROFILE_MAX_DEPTH = 500.0   # depth-axis limit, m; None = full 969 m column
# ------------------------------------------------------------------------

import dask
import numpy as np

import dbof.preprocessing.calculate_fields as CF
import dbof.preprocessing.calculate_fields_at_depth as CFAD
from dbof.plotting import depth_figures as dfig
from dbof.plotting.field_cmaps import load_field_cmaps
import dbof.utils.native_gradient as NG
from dbof.preprocessing.vertical_helpers import (
    _interp_w_to_tracer_levels, _vertical_derivative,
)
from dbof.tiles import tile_utils
from dbof.tiles.tile_mapping import rect_ij_to_tile
from dbof.global_dataset_creation.subset_definitions import (
    get_compute_fn, get_subset_definition, expand_channels_with_suffixes,
)

# tile_utils sets the Agg backend when it is imported (it writes QA PNGs
# on headless nodes), so switch back to inline AFTER the dbof imports or
# no figure in this notebook will render.
%matplotlib inline
import matplotlib.pyplot as plt

CMAP_CFG, DIVERGING = load_field_cmaps()

# Channel list straight from the pipeline's own definition -- if the
# subset gains a channel, this notebook picks it up without an edit.
defn = get_subset_definition(PIPELINE, SUBSET)
CHANNELS = expand_channels_with_suffixes(
    defn["compute_features_channels"], list(LEVELS),
    defn.get("extra_channels"),
)
print(f"subset   : {SUBSET}")
print(f"channels : {CHANNELS}")

## Section 2 — Load the tile and compute the fields

We do **not** run `generate-global` here.  That would compute the whole
planet in order to look at one place.

Instead this notebook works on **one tile** — the 720 × 720 × 51 block
the `dbof.tiles` workflow already defines: one LLC face, the full water
column, about 1400 × 1400 km, centred on the region's anchor.  A tile is
*exactly one chunk* of the depth store, so loading it costs one S3 GET
per variable (~106 MB per 3D field).  Tiles are 720-aligned and faces
are 6 × 720 wide, so a tile can never straddle two faces.

Then the **production** compute function for this subset runs on it,
and internally applies the four depth strategies.  Same code as
production, one tile's worth of data.

One thing this costs us: the tile's xgcm grid has **no face
connections**, so cells near the boundary have no neighbours and their
horizontal gradients are wrong.  That rim is NaN'd, using the per-field
widths `tiles/field_registry.py` already records (0 for purely vertical
fields, 1 for staggered interpolation, 3 for gradient and Jacobian
chains).

**A tile samples the region, it does not cover it.**  "Gulf Stream"
here means the ~1400 km tile around 60°W / 37°N — not the whole
80–40°W box the surface notebooks use as a row.


In [ ]:
# Anchor -> rect pixel -> the tile that contains it.
S3 = tile_utils._resolve_s3_source(None)
ANCHOR_LON, ANCHOR_LAT = dfig.region_anchor(REGION)

i_rect, j_rect = tile_utils.latlon_to_rect_ij(ANCHOR_LON, ANCHOR_LAT, S3)
tile = rect_ij_to_tile(i_rect, j_rect)
print(f"region : {REGION} anchored at ({ANCHOR_LON}, {ANCHOR_LAT})")
print(f"tile   : idx {tile.tile_idx}, face {tile.face_idx}, "
      f"j={tile.j_face_slice}, i={tile.i_face_slice}")

# Load the tile + its grid, then merge and build a LOCAL xgcm grid.
ds_grid = tile_utils._load_grid_for_tile(S3, tile)
ds_raw = tile_utils._load_tracers_for_tile(S3, DATE, tile, RAW_VARS)
ds_merge, xgrid = tile_utils._build_tile_context(ds_raw, ds_grid)

XC, YC = dfig.tile_coords(ds_grid)
LAND = dfig.tile_land_mask(ds_grid)
print(f"extent : lon [{XC.min():.2f}, {XC.max():.2f}], "
      f"lat [{YC.min():.2f}, {YC.max():.2f}], "
      f"land {100 * LAND.mean():.1f}%")

### The finals, and the intermediates the figures need

`get_compute_fn("DEPTH", SUBSET)` is the production entry point — the
same callable `generate-global` dispatches to — so the finals below are
the pipeline's own numbers.

The **intermediates** are a different matter: the global products never
store them, so they are recomputed here from the same `ds_merge` the
finals came from.  That is deliberate — it means each figure's chain
shows the actual steps, not a reconstruction.


In [ ]:
store = get_compute_fn(PIPELINE, SUBSET)(ds_merge, xgrid, CHANNELS)
print(f"computed : {sorted(store)}")

mld = CFAD.mixed_layer_depth(ds_merge)

# Model-axis velocity at tracer points: interpolated but NOT rotated.
# This is the "before" of the CS/SN rotation, and is not a pipeline
# output -- it exists only so the figures can show what rotating does.
U_model = xgrid.interp(ds_merge.U, 'X', boundary='fill')
V_model = xgrid.interp(ds_merge.V, 'Y', boundary='fill')

PROFILE_3D = {
    "Theta": ds_merge["Theta"],
    "Salt": ds_merge["Salt"],
    "U_model": U_model,
    "V_model": V_model,
}
live = dfig.compute_levels(PROFILE_3D, ds_merge, mld=mld, levels=LEVELS)

In [ ]:
# How wide the invalid rim is for this subset, straight from the tile
# registry (0 here: nothing in this chain takes a horizontal gradient).
EDGE_MARGIN = dfig.edge_margin_for(
    list(defn["compute_features_channels"])
    + list(defn.get("extra_channels") or []))

# NaN that rim, mask land with the surface hFacC (what production does),
# and reshape into the {base: {level: (x, y, arr)}} the figures take.
level_arrays = dfig.pack_tile_levels(
    {**live, **store}, XC, YC, edge_margin=EDGE_MARGIN,
    land_mask=LAND, levels=LEVELS)

In [ ]:
# Five ocean columns, seeded and spread across the tile, reused by every
# field in this notebook so the profile panels are comparable.
POINTS = dfig.pick_profile_points(
    LAND, n=N_PROFILES, edge_margin=max(EDGE_MARGIN, 1),
    seed=PROFILE_SEED)

# Full water column at those five points -- a few hundred numbers per
# field, so this is cheap next to the maps.
PROFILES, DEPTH_M = dfig.sample_profiles(PROFILE_3D, ds_merge, POINTS)

# MLD at each point, to mark on the profiles.
MLD_AT_POINTS = (
    [level_arrays["mixed_layer_depth"]["sfc"][2][j, i] for j, i in POINTS]
    if "mixed_layer_depth" in level_arrays else None)

## Section 3 — Subset: `native_fields`

| Channel | Kind |
|---|---|
| `Theta_{sfx}`, `Salt_{sfx}` | model output, tracer points |
| `U_{sfx}`, `V_{sfx}` | interpolated to tracer points + CS/SN rotated |
| `W_{sfx}` | model output on `k_l`, interpolated to tracer levels |
| `Eta_sfc` | model output, inherently 2D |


## Section 4 — Field & dependency table

| FIELD | UNITS | EQUATION | DEPENDS ON | CODE |
|---|---|---|---|---|
| `Theta_{sfx}`, `Salt_{sfx}` | °C, psu | passthrough | model | — |
| `U_model`, `V_model` | m s⁻¹ | interp to tracer points, NO rotation | U (i_g), V (j_g) | `xgcm.Grid.interp` |
| `U_{sfx}` | m s⁻¹ | u_east = u·CS − v·SN | U_model, V_model, CS, SN | `calculate_fields.geographic_velocity` |
| `V_{sfx}` | m s⁻¹ | v_north = u·SN + v·CS | U_model, V_model, CS, SN | `calculate_fields.geographic_velocity` |
| `W_{sfx}` | m s⁻¹ | W_c = ½(W[k] + W[k+1]) | W on `k_l` | `vertical_helpers._interp_w_to_tracer_levels` |
| `Eta_sfc` | m | passthrough | model | — |

**Rotation is the whole point of this notebook.**  The model x/y axes
have arbitrary orientation that varies within and across LLC faces, so
`U_model` is NOT eastward.  Comparing the `U_model` column with the
`U` column shows how much of the flow the rotation moves between
components — on a face whose axes are near-geographic they look alike;
on a rotated face they do not.

**Tile edge rim:** the staggered→tracer interpolation costs one cell
(`edge_margin = 1` in `field_registry`), so a 1-cell border is NaN.

**Gradient artifacts:** none — nothing here differentiates.  W is
interpolated vertically (an average of two levels), which smooths but
does not cancel.

**W at k_l = 0** is the surface interface and is ~0 by construction, so
the `sfc` row of W is near-empty by design, not a bug.


## Section 5 — Per-field validation

Three figures per field.

**Figure 1 — maps.**  Columns are the dependency chain, raw → final.
Rows are the four depth levels over the whole tile, then the same four
zoomed to a 200 × 200 km box; the crimson square on the whole-tile rows
is where the zoom is.  One colour scale per column, shared by every row
including the zooms, so nothing changes colour when you look closer.

Fields that **do not vary with depth** get two rows instead of eight —
whole tile and zoom.  `mixed_layer_depth` and `ml_heat_content` are
both integrals over the entire water column, so four identical depth
rows would say nothing.  Their 3D chain inputs are shown at the surface
in those figures, and the title says so.

**Figure 2 — PDFs.**  Same columns; four rows, the whole tile at each
level.  Bins are shared down a column, so reading a column top to
bottom shows how the distribution changes with depth.  The zoom boxes
are deliberately absent — too few cells to make an honest histogram.

**Figure 3 — profiles.**  Five ocean columns, spread across the tile
and fixed by a seed so every field profiles the same water.  The
leftmost panel shows where they are, as numbered colour-coded ×; then
one panel per 3D field in the chain, with the **surface at the top and
depth increasing downward**.  Dashed horizontal lines mark each
location's mixed-layer depth.  The location numbers repeat in the
legend, so the five are distinguishable without relying on colour.

("Grid" in the function names below means the rows × columns array of
panels — not the model's Arakawa C-grid, which is `docs/Grid.md`.)


In [ ]:
# Section 5 helpers: one call per figure, shared by every field.
CHAINS = {
    "Theta": ["Theta"],
    "Salt": ["Salt"],
    "U": ["U_model", "V_model", "U"],
    "V": ["U_model", "V_model", "V"],
    "W": ["W"],
    "Eta": ["Eta"],
}
LOG_FIELDS = set()

# Fields with no depth dependence -- integrals over the whole column.
# Their figures collapse to 2 rows (whole tile + zoom) / 1 PDF row.
DEPTH_INVARIANT = {"Eta"}


def figure1_maps(field):
    """Figure 1: chain across columns, depth down rows."""
    flat = field in DEPTH_INVARIANT
    note = (" | depth-invariant: 3D inputs shown at the surface"
            if flat else "")
    dfig.depth_map_grid(
        CHAINS[field], level_arrays, CMAP_CFG,
        region=REGION,
        levels=("sfc",) if flat else LEVELS,
        row_labels=(("whole tile", f"{2 * ZOOM_HALF_KM:.0f} km zoom")
                    if flat else None),
        diverging_cmaps=DIVERGING,
        log_scale_channels=LOG_FIELDS,
        zoom_half_km=ZOOM_HALF_KM,
        suptitle=(f"Figure 1 — {field} | {REGION} tile | columns = "
                  f"dependency chain, rows = depth{note}"),
    )
    plt.show()


def figure2_pdfs(field):
    """Figure 2: PDFs, chain across columns, depth down rows."""
    flat = field in DEPTH_INVARIANT
    dfig.depth_pdf_grid(
        CHAINS[field], level_arrays, CMAP_CFG,
        levels=("sfc",) if flat else LEVELS,
        row_labels=("whole tile",) if flat else None,
        log10_fields=LOG_FIELDS,
        suptitle=(f"Figure 2 — {field} | {REGION} tile | density; "
                  f"land + rim NaNs dropped; bins shared down each column"),
    )
    plt.show()


def figure3_profiles(field):
    """Figure 3: depth profiles at the five fixed locations."""
    dfig.depth_profile_grid(
        CHAINS[field], PROFILES, DEPTH_M, CMAP_CFG,
        points=POINTS, level_arrays=level_arrays, region=REGION,
        mld_at_points=MLD_AT_POINTS,
        diverging_cmaps=DIVERGING,
        log_scale_channels=LOG_FIELDS,
        max_depth=PROFILE_MAX_DEPTH,
        suptitle=(f"Figure 3 — {field} | {REGION} tile | profiles at "
                  f"{len(POINTS)} locations; surface at top"),
    )
    plt.show()

In [ ]:
# Safety net: every field named in a chain must actually have been
# computed, or the figure call fails deep inside matplotlib.
_missing = sorted({f for c in CHAINS.values() for f in c}
                  - set(level_arrays))
assert not _missing, f"chain fields never computed: {_missing}"
print(f"chains OK : {len(CHAINS)} fields, "
      f"{len({f for c in CHAINS.values() for f in c})} distinct columns")

### Θ — potential temperature

**passthrough** [°C]

Sanity baseline.  Expect the Gulf Stream front as a sharp lateral gradient, warm to the south, and monotonic cooling with depth across the four rows.

In [ ]:
figure1_maps("Theta")

In [ ]:
figure2_pdfs("Theta")

In [ ]:
figure3_profiles("Theta")

### S — salinity

**passthrough** [psu]

Same front, opposite side of the density balance: salty subtropical water south, fresher slope water north.

In [ ]:
figure1_maps("Salt")

In [ ]:
figure2_pdfs("Salt")

In [ ]:
figure3_profiles("Salt")

### U — eastward velocity

**u_east = u·CS − v·SN**  [m s⁻¹]

Compare the `U_model` column with the `U` column: the difference is entirely the grid rotation.  Expect the jet as a coherent eastward band that weakens with depth.

In [ ]:
figure1_maps("U")

In [ ]:
figure2_pdfs("U")

In [ ]:
figure3_profiles("U")

### V — northward velocity

**v_north = u·SN + v·CS**  [m s⁻¹]

The meandering component.  On a rotated face `V_model` and `V` can look very different — that is the rotation working, not an error.

In [ ]:
figure1_maps("V")

In [ ]:
figure2_pdfs("V")

In [ ]:
figure3_profiles("V")

### W — vertical velocity

**W_c = ½(W[k] + W[k+1])**  [m s⁻¹]

Two orders of magnitude smaller than U/V and much noisier. The `surface` row is ~0 by construction (k_l = 0 is the rigid-lid interface).  W is the noisiest 3D field in the model and feeds `ertel_pv` — worth remembering there.

In [ ]:
figure1_maps("W")

In [ ]:
figure2_pdfs("W")

In [ ]:
figure3_profiles("W")

### η — sea-surface height

**passthrough** [m]

Inherently 2D, so two rows.  The jet is the SSH front; `ug`/`vg` in the frontogenesis notebook are derived from its gradient.

In [ ]:
figure1_maps("Eta")

In [ ]:
figure2_pdfs("Eta")

In [ ]:
figure3_profiles("Eta")

## Section 6 — Literature comparison

**PENDING — nothing to build here yet.**

The comparison figure is chosen *after* the literature figure is, not
before.  Once LH picks a paper figure and drops the PNG into
`../literature_figures/` (naming convention
`{field(s)}_{Citation}_{description}.png`), we decide which of our
panels belongs beside it and add a subsection here — one subsection per
reference, using `dbof.plotting.literature_comparison.side_by_side`.

Leave this section as-is until then.


## Summary — did every channel come out sane?

Coverage and range for each channel at each level, then the physical
checks that are worth failing loudly on.


In [ ]:
# Coverage + range per field per level.
print(f"{'field':<22}{'level':<10}{'finite %':>9}"
      f"{'min':>14}{'max':>14}")
print("-" * 69)
for field in sorted(level_arrays):
    for lev in LEVELS:
        arr = level_arrays[field][lev][2]
        finite = np.isfinite(arr)
        pct = 100.0 * finite.mean()
        lo = np.nanmin(arr) if finite.any() else np.nan
        hi = np.nanmax(arr) if finite.any() else np.nan
        print(f"{field:<22}{lev:<10}{pct:>8.1f}%{lo:>14.4g}{hi:>14.4g}")

In [ ]:
# Physical checks.  These assert -- a red cell here is a real problem.
u = level_arrays["U"]["sfc"][2]
v = level_arrays["V"]["sfc"][2]
um = level_arrays["U_model"]["sfc"][2]
vm = level_arrays["V_model"]["sfc"][2]
w_sfc = level_arrays["W"]["sfc"][2]
th_s = level_arrays["Theta"]["sfc"][2]
th_d = level_arrays["Theta"]["mld"][2]

CHECKS = [
    ("rotation preserves speed (it is a rotation, not a rescale)",
     np.allclose(np.sqrt(u**2 + v**2), np.sqrt(um**2 + vm**2),
                 rtol=1e-4, equal_nan=True),
     f"max |dspeed| = "
     f"{np.nanmax(np.abs(np.sqrt(u**2+v**2)-np.sqrt(um**2+vm**2))):.2e}"),
    ("surface speed below 4 m/s",
     np.nanmax(np.sqrt(u**2 + v**2)) < 4.0,
     f"max = {np.nanmax(np.sqrt(u**2 + v**2)):.2f} m/s"),
    ("W at k_l=0 is ~0 (rigid lid)",
     np.nanmax(np.abs(w_sfc)) < 1e-6,
     f"max |W_sfc| = {np.nanmax(np.abs(w_sfc)):.2e} m/s"),
    ("Theta in a physical range",
     (np.nanmin(th_s) > -3.0) and (np.nanmax(th_s) < 40.0),
     f"[{np.nanmin(th_s):.1f}, {np.nanmax(th_s):.1f}] degC"),
    ("water at the MLD is no warmer than the surface (median)",
     np.nanmedian(th_d) <= np.nanmedian(th_s) + 0.05,
     f"median sfc {np.nanmedian(th_s):.2f}, mld {np.nanmedian(th_d):.2f}"),
    ("Salt in a physical range",
     (np.nanmin(level_arrays["Salt"]["sfc"][2]) > 25.0)
     and (np.nanmax(level_arrays["Salt"]["sfc"][2]) < 42.0),
     f"[{np.nanmin(level_arrays['Salt']['sfc'][2]):.1f}, "
     f"{np.nanmax(level_arrays['Salt']['sfc'][2]):.1f}] psu"),
]

failures = []
for name, ok, detail in CHECKS:
    print(f"{'OK  ' if ok else 'FAIL'}  {name}  ({detail})")
    if not ok:
        failures.append(name)
assert not failures, f"physical checks failed: {failures}"
print("\nAll physical checks passed.")

---

### Cross-references

- **MLD**, which sets the `_mld` / `_mld_mean` rows here —
  `stratification.ipynb`.
- **The rotation machinery** (CS/SN, why vectors are never
  passthroughs) — `docs/Gradients.md` and
  `surface_fields/native_fields.ipynb`.
- **W feeds `ertel_pv`** — its noise matters there, see
  `ertel_pv.ipynb`.
